# transistor：今週の実験をまとめたMSE版

**実行はユーザーが行います。このファイルは未実行です。**

1. 前処理・回転拡張の確認
2. 正常画像170枚／検証43枚で4種類のVAEを比較（各100エポック）
3. 正常213枚でHybridを通常100エポック・回転100エポック・通常6,700更新の3条件で学習
4. 回転なし100枚／0度なしランダム回転100枚のAUROC・復元・誤差マップを比較

合計7モデルを**新規にMSEで学習**します。SSE版のモデルは読み込まず、既存ファイルは変更しません。

$$L=\underbrace{\frac1B\sum_{b=1}^B\frac1{16384}\sum_p(x_{bp}-\hat x_{bp})^2}_{\mathrm{MSE}}+
\beta\underbrace{\frac1B\sum_{b=1}^B\frac12\sum_{i=1}^{64}(\mu_{bi}^2+\sigma_{bi}^2-\log\sigma_{bi}^2-1)}_{\text{KL：潜在次元は合計、バッチは平均}}.$$

**ユーザー指定によりβは0→1のままです（16,384で割りません）。** 同じβ時点ではSSE版よりKLが相対的に16,384倍強くなるため、復元低下や潜在表現が入力情報を使わなくなる可能性も検証対象です。MSE表記への単なる換算とは異なります。

Sparse VAEの実装は含みません。復元モデル選定の指標と、異常検知のAUROCを区別します。

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from io import BytesIO
from datetime import datetime
from zoneinfo import ZoneInfo
import copy, hashlib, json, math, platform, random, time, uuid
import importlib.metadata
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from torchvision.transforms import functional as TF
from skimage.metrics import structural_similarity as ssim
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p/'memory/memory.md').is_file() and (p/'data/transistor/train/good').is_dir()), None)
if ROOT is None: raise FileNotFoundError('リポジトリ配下で開いてください。')
OUTPUT_BASE = ROOT/'saved_models/transistor_mse_integrated_20261106'
RUN_FULL_TRAINING = True
TRAINING_RUN_ID = None  # 再学習せず閲覧：Falseと、保存先の実行IDを指定。Noneなら最新の完了済み実行。
SEED = 42
SPLIT_SEED = 42
ASSIGNMENT_SEED = 42
EPOCHS = 100
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 0.001
LATENT_DIM = 64
ANNEALING_EPOCHS = 50
FINAL_BETA = 1.0  # ユーザー指定。画素数による係数の補正はしない。
SSIM_WINDOW = 11
TOP_K = 130
ANGLES = tuple(range(0,360,36))
TEST_ANGLES = tuple(range(36,360,36))
SOURCE_SIZE = (1024,1024)
INPUT_SIZE = (128,128)
PIXELS = math.prod(INPUT_SIZE)
MODEL_KEYS = ('FC','Shallow3','Deep5','Hybrid')
ROTATION_KEYS = ('A_normal100','B_rotate100','A_normal_matched')
LABELS = {'FC':'FC','Shallow3':'CNN Shallow3','Deep5':'CNN Deep5','Hybrid':'Hybrid CNN＋FC',
          'A_normal100':'通常学習100エポック','B_rotate100':'回転拡張100エポック',
          'A_normal_matched':'通常学習6,700更新'}
COLORS = {'FC':'#777744','Shallow3':'#287c80','Deep5':'#446c9b','Hybrid':'#b45342',
          'A_normal100':'#777777','B_rotate100':'#d98b27','A_normal_matched':'#446c9b'}
CATEGORIES = {'good':'正常','bent_lead':'端子の曲がり','cut_lead':'端子の切断',
              'damaged_case':'本体の損傷','misplaced':'配置ずれ'}
SCENARIOS = {'unrotated':'回転なし','random_rotated':'ランダム回転（0度なし）'}
MAX_VALIDATION_IMAGES = 6  # Noneなら43枚すべて
MAX_IMAGES_PER_CATEGORY = 1  # Noneなら各種類すべて。AUROCの計算枚数には影響しない。
VISUAL_PATHS = None  # 例：['good/000.png','cut_lead/000.png']。指定時はこちらを優先。
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(4,max(1,os.cpu_count() or 1)))
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
fonts={f.name for f in font_manager.fontManager.ttflist}
plt.rcParams['font.family']=next((f for f in ['Meiryo','Yu Gothic','Noto Sans CJK JP','MS Gothic'] if f in fonts),'sans-serif')
plt.rcParams['axes.unicode_minus']=False

def new_id(): return datetime.now(ZoneInfo('Asia/Tokyo')).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
def sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def json_hash(v): return hashlib.sha256(json.dumps(v,ensure_ascii=False,sort_keys=True).encode('utf8')).hexdigest()
def write_json(p,v): Path(p).write_text(json.dumps(v,ensure_ascii=False,indent=2,allow_nan=False)+'\n',encoding='utf8')
def read_json(p): return json.loads(Path(p).read_text(encoding='utf8'))
def seed_everything(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
def copy_cpu_state(m): return {k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
def state_digest(state):
    h=hashlib.sha256()
    for k,v in state.items():
        a=v.detach().cpu().contiguous();h.update(k.encode()+str(a.dtype).encode()+a.numpy().tobytes())
    return h.hexdigest()
print('実行デバイス：',DEVICE,'／正式学習：',RUN_FULL_TRAINING)

## 1. データと前処理
元画像全体1024×1024 → 128×128へLANCZOS縮小 → グレースケール → 0〜1。
回転時だけ、高解像度で反射補完・最近傍回転・中央1024×1024の取り出しを先に行います。
0度は元画像をコピーします。反射で模様が複製される可能性、切り取りの影響は残ります。
モデル選定の170／43分割にはtrain/goodのみを使います。testの正常60枚もモデル選定には使いません。

In [ ]:
final_transform=transforms.Compose([
    transforms.Resize(INPUT_SIZE,interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.Grayscale(num_output_channels=1),
])
to_tensor=transforms.ToTensor()
def rotation_pad_width(size):
    return math.ceil((math.hypot(*size)-min(size))/2)+4

def rotate_with_reflect(image,angle,verify_support=False):
    if angle==0: return image.copy()
    width,height=image.size
    pad=rotation_pad_width(image.size)
    pixels=np.asarray(image)
    extended=Image.fromarray(np.pad(pixels,((pad,pad),(pad,pad),(0,0)),mode='reflect'))
    box=(pad,pad,pad+width,pad+height)
    options=dict(interpolation=transforms.InterpolationMode.NEAREST,expand=False)
    out=TF.rotate(extended,angle=angle,fill=255,**options).crop(box)
    mask=TF.rotate(Image.new('L',image.size,255),angle=angle,fill=0,**options)
    original_part=TF.rotate(image,angle=angle,fill=255,**options)
    valid=np.asarray(mask)>0
    result=np.asarray(out).copy()
    result[valid]=np.asarray(original_part)[valid]
    if verify_support:
        support=TF.rotate(Image.new('L',extended.size,255),angle=angle,fill=0,**options).crop(box)
        assert np.all(np.asarray(support)==255),'パディングが不足しています。'
        assert np.array_equal(result[valid],np.asarray(original_part)[valid])
    return Image.fromarray(result)


def load_images(paths):
    images,tensors,records=[],[],[]
    for path in paths:
        raw=path.read_bytes()
        with Image.open(BytesIO(raw)) as im: image=im.convert('RGB').copy()
        if image.size!=SOURCE_SIZE: raise ValueError(f'想定外の画像サイズ：{path} {image.size}')
        tensor=to_tensor(final_transform(image))
        images.append(image);tensors.append(tensor)
        records.append({'path':path.relative_to(ROOT).as_posix(),'file_sha256':hashlib.sha256(raw).hexdigest(),
                        'input_sha256':hashlib.sha256(tensor.numpy().tobytes()).hexdigest(),'source_size':list(image.size)})
    if not records: raise FileNotFoundError('画像がありません。')
    return images,torch.stack(tensors),records

train_paths=sorted((ROOT/'data/transistor/train/good').glob('*.png'),key=lambda p:p.name)
assert len(train_paths)==213
train_images,all_train,train_manifest=load_images(train_paths)
permutation=np.random.default_rng(SPLIT_SEED).permutation(213)
validation_indices=np.sort(permutation[:43]);training_indices=np.sort(permutation[43:])
selection_train=all_train[training_indices].contiguous()
selection_val=all_train[validation_indices].contiguous()
split_manifest={'seed':SPLIT_SEED,'train':[train_manifest[i] for i in training_indices],
                'validation':[train_manifest[i] for i in validation_indices]}
assert len(selection_train)==170 and len(selection_val)==43
for key in ['file_sha256','input_sha256']:
    assert not {r[key] for r in split_manifest['train']}&{r[key] for r in split_manifest['validation']}
assert torch.isfinite(all_train).all() and 0<=all_train.min()<=all_train.max()<=1
print('モデル選定：170枚学習／43枚検証、回転比較：213枚をすべて学習')

In [ ]:
# 画像の表示だけ。モデル学習・評価はまだ行いません。
fig,axes=plt.subplots(2,5,figsize=(13,6),constrained_layout=True)
for j in range(5):
    axes[0,j].axis('off')
    if j==0:
        axes[0,j].imshow(all_train[0,0],cmap='gray',vmin=0,vmax=1)
        axes[0,j].set_title('通常学習：元画像1枚')
    angle=ANGLES[j]
    preview=to_tensor(final_transform(rotate_with_reflect(train_images[0],angle,verify_support=True)))
    axes[1,j].imshow(preview[0],cmap='gray',vmin=0,vmax=1)
    axes[1,j].set_title(f'回転拡張：{angle}度');axes[1,j].axis('off')
fig.suptitle('学習入力の比較（回転の先頭5方向）');plt.show();plt.close(fig)
fig,axes=plt.subplots(2,5,figsize=(13,6),constrained_layout=True)
for ax,angle in zip(axes.flat,ANGLES):
    image=to_tensor(final_transform(rotate_with_reflect(train_images[0],angle)))
    ax.imshow(image[0],cmap='gray',vmin=0,vmax=1);ax.set_title(f'{angle}度');ax.axis('off')
fig.suptitle('回転拡張の最終入力10方向');plt.show();plt.close(fig)

## 2. 既存と同じ4種類のVAE
以下のモデル定義は、既存の復元モデル比較ノートブックから複製したものです。
潜在64、log分散の制限−10〜10、学習時のサンプリング・評価時の平均推論を維持します。
構造が異なる4モデルの初期重み自体は同一にできません。seedと学習画像順を共通にします。
回転比較のHybrid 3条件は、同じ初期重みからそれぞれ独立に学習します。

In [ ]:
class VAE_FC(nn.Module):
    """
    全結合層(FC)のみを用いて特徴抽出と復元を行うベースラインモデル
    """
    def __init__(self, latent_dim=64):
        super(VAE_FC, self).__init__()
        
        # --- エンコーダ: 段階的に次元を圧縮 (16384 -> 1024 -> 256 -> 128) ---
        self.encoder = nn.Sequential(
            nn.Linear(16384, 1024), nn.ReLU(),
            nn.Linear(1024, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )
        
        # 潜在空間へのマッピング
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # --- デコーダ: 段階的に次元を復元 (64 -> 128 -> 256 -> 1024 -> 16384) ---
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 1024), nn.ReLU(),
            nn.Linear(1024, 16384), nn.Sigmoid()
        )

    def forward(self, x, sample=True):
        # 画像を1次元に平坦化
        h = torch.flatten(x, start_dim=1)
        h = self.encoder(h)
        
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)
        
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std if sample else mu
        
        recon = self.decoder(z)
        
        # 画像の形状に戻して返す
        return recon.view(-1, 1, 128, 128), mu, logvar


class VAE_CNN_Shallow3(nn.Module):
    """
    浅い3層の畳み込み層を用いた軽量VAEモデル
    """
    def __init__(self, latent_dim=64):
        super(VAE_CNN_Shallow3, self).__init__()
        
        # --- エンコーダ: 畳み込み層で空間情報を保ちながら圧縮 (128 -> 64 -> 32 -> 16) ---
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        
        # 64チャンネル * 16 * 16 = 16384次元
        self.fc_mu = nn.Linear(64 * 16 * 16, latent_dim)
        self.fc_logvar = nn.Linear(64 * 16 * 16, latent_dim)
        
        self.decoder_fc = nn.Linear(latent_dim, 64 * 16 * 16)
        
        # --- デコーダ: 逆畳み込み層で元の画像サイズへ復元 ---
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, sample=True):
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)
        
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        
        # 潜在変数の発散(KLD爆発)を防ぐ安全装置
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)
        
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std if sample else mu
        
        h_recon = self.decoder_fc(z)
        h_recon = h_recon.view(-1, 64, 16, 16) 
        recon = self.decoder_conv(h_recon)
        
        return recon, mu, logvar


class VAE_CNN_Deep5(nn.Module):
    """
    5層の深い畳み込み層を用いたVAEモデル
    """
    def __init__(self, latent_dim=64):
        super(VAE_CNN_Deep5, self).__init__()
        
        # --- エンコーダ: 畳み込み層を5層重ねてさらに深く圧縮 (128 -> 64 -> 32 -> 16 -> 8 -> 4) ---
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )
        
        # 最終層のサイズ: 256チャンネル * 4ピクセル * 4ピクセル = 4096次元
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)
        
        # デコーダ用の全結合層
        self.decoder_fc = nn.Linear(latent_dim, 256 * 4 * 4)
        
        # --- デコーダ: 逆畳み込み層を5層重ねて画像サイズを復元 (4 -> 8 -> 16 -> 32 -> 64 -> 128) ---
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, sample=True):
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)
        
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        
        # 潜在変数の発散(KLD爆発)を防ぐ安全装置
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)
        
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std if sample else mu
        
        h_recon = self.decoder_fc(z)
        h_recon = h_recon.view(-1, 256, 4, 4) 
        recon = self.decoder_conv(h_recon)
        
        return recon, mu, logvar


class VAE_Hybrid_CNN_FC(nn.Module):
    def __init__(self, latent_dim=64):
        super(VAE_Hybrid_CNN_FC, self).__init__()

        # -------------------------------------------------------------
        # 【エンコーダ前半】 CNN_Deep5（空間圧縮: 128x128 -> 4x4）
        # -------------------------------------------------------------
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU()
        )

        # -------------------------------------------------------------
        # 【エンコーダ後半】 FC_Only（段階的圧縮: 4096次元 -> 128次元）
        # -------------------------------------------------------------
        self.encoder_fc = nn.Sequential(
            nn.Linear(256 * 4 * 4, 1024), nn.ReLU(),
            nn.Linear(1024, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )

        # 潜在空間へのマッピング（128次元 -> 64次元）
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # -------------------------------------------------------------
        # 【デコーダ前半】 FC_Only（段階的復元: 64次元 -> 4096次元）
        # -------------------------------------------------------------
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 1024), nn.ReLU(),
            # ※CNNへ繋ぐため、最後はSigmoidではなくReLUを使用します
            nn.Linear(1024, 256 * 4 * 4), nn.ReLU()
        )

        # -------------------------------------------------------------
        # 【デコーダ後半】 CNN_Deep5（空間復元: 4x4 -> 128x128）
        # -------------------------------------------------------------
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),

            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),

            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, sample=True):
        # 1. CNNによる特徴抽出
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)

        # 2. FCによる段階的圧縮
        h = self.encoder_fc(h)

        # 3. 潜在変数の計算と安全装置（クランプ）
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)

        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std if sample else mu

        # 4. FCによる段階的復元
        h_recon = self.decoder_fc(z)
        h_recon = h_recon.view(-1, 256, 4, 4) # 4階テンソルに戻す

        # 5. CNNによる画像復元
        recon = self.decoder_conv(h_recon)

        return recon, mu, logvar

MODEL_FACTORIES = {'FC': VAE_FC, 'Shallow3': VAE_CNN_Shallow3,
                   'Deep5': VAE_CNN_Deep5, 'Hybrid': VAE_Hybrid_CNN_FC}
MODEL_DESCRIPTIONS = {
    'FC': '全結合：16384→1024→256→128→潜在64',
    'Shallow3': 'CNN3段：128→64→32→16、最終特徴を潜在64へ',
    'Deep5': 'CNN5段：128→64→32→16→8→4、最終特徴を潜在64へ',
    'Hybrid': 'CNN5段＋全結合：4096→1024→256→128→潜在64',
}

MODEL_DEFINITION_SHA = '51de270b653e1501ec08504244cc88c754049ebeee11eb35c5a31a6a9328797e'


In [ ]:
model_rows=[]
for key in MODEL_KEYS:
    seed_everything(SEED)
    model=MODEL_FACTORIES[key](latent_dim=LATENT_DIM)
    model_rows.append({'model':key,'構造':MODEL_DESCRIPTIONS[key],
                       'パラメータ数':sum(p.numel() for p in model.parameters())})
    del model
model_table=pd.DataFrame(model_rows);display(model_table)

## 3. MSE版の損失と7条件の学習計画

- `loss_terms`はMSE・KL・総損失を**画像ごと**に返し、最適化時にバッチ平均を取ります。
- βはエポック0始まりで `min(epoch_index / 50, 1)`。最初の1周は0、51周目以降は1です。
- 更新回数合わせは、回転学習の1周67更新を基準にβも一致させます。
- 通常100周は700更新、回転100周と更新回数合わせは6,700更新です。
- 通常6,700更新は957周＋1バッチ。画像順・端数バッチ・総提示枚数・BatchNorm統計まで同一にはなりません。
- βが学習中に変化するため、総損失曲線は固定された一つの目的関数の推移ではありません。

In [ ]:
def loss_terms(reconstruction,inputs,mu,logvar,beta):
    mse=(reconstruction-inputs).square().flatten(1).mean(dim=1)
    kl=0.5*(mu.square()+logvar.exp()-logvar-1).sum(dim=1)
    total=mse+float(beta)*kl
    return mse,kl,total

def beta_at_update(update_index,schedule_updates_per_epoch):
    return FINAL_BETA*min((update_index//schedule_updates_per_epoch)/ANNEALING_EPOCHS,1.0)

PLAN=[]
for key in MODEL_KEYS:
    PLAN.append({'task':'selection_'+key,'group':'selection','model':key,'dataset':'selection_train',
                 'target_updates':600,'schedule_updates_per_epoch':6,'label':LABELS[key]})
PLAN.extend([
    {'task':'A_normal100','group':'rotation','model':'Hybrid','dataset':'original',
     'target_updates':700,'schedule_updates_per_epoch':7,'label':LABELS['A_normal100']},
    {'task':'B_rotate100','group':'rotation','model':'Hybrid','dataset':'rotated',
     'target_updates':6700,'schedule_updates_per_epoch':67,'label':LABELS['B_rotate100']},
    {'task':'A_normal_matched','group':'rotation','model':'Hybrid','dataset':'original',
     'target_updates':6700,'schedule_updates_per_epoch':67,'label':LABELS['A_normal_matched']},
])
assert math.ceil(len(selection_train)/BATCH_SIZE)*EPOCHS==600
assert math.ceil(len(all_train)/BATCH_SIZE)*EPOCHS==700
assert math.ceil(len(all_train)*len(ANGLES)/BATCH_SIZE)*EPOCHS==6700
display(pd.DataFrame(PLAN))

In [ ]:
# 学習用とモデル選定用のデータだけで実行識別情報を作ります。
CONTRACT={
    'experiment':'transistor_mse_integrated_20261106_v1','model_definition_sha256':MODEL_DEFINITION_SHA,
    'plan':PLAN,'train_manifest_sha256':json_hash(train_manifest),'split_sha256':json_hash(split_manifest),
    'seed':SEED,'batch_size':BATCH_SIZE,'latent_dim':LATENT_DIM,'optimizer':'Adam',
    'learning_rate':LEARNING_RATE,'beta_final':FINAL_BETA,'annealing_epochs':ANNEALING_EPOCHS,
    'loss':'pixel mean MSE, latent sum KL, batch mean total=MSE+beta*KL',
    'kl_rescale_by_pixels':False,'input_size':list(INPUT_SIZE),'source_size':list(SOURCE_SIZE),
    'training_angles':list(ANGLES),'padding':'reflect','padding_width':rotation_pad_width(SOURCE_SIZE),
    'rotation':'torchvision PIL nearest, keep frame, preserve original support',
    'preprocessing':'full RGB, high-resolution rotation if requested, LANCZOS resize, grayscale, ToTensor',
    'training_inference':'posterior sample','evaluation_inference':'posterior mean',
}
CONTRACT_SHA=json_hash(CONTRACT)

def mean_reconstruction_metrics(model,inputs):
    model.eval();sums=np.zeros(2);count=0
    with torch.inference_mode():
        for start in range(0,len(inputs),EVAL_BATCH_SIZE):
            x=inputs[start:start+EVAL_BATCH_SIZE].to(DEVICE)
            out,mu,lv=model(x,sample=False)
            mse,kl,_=loss_terms(out,x,mu,lv,0)
            if not torch.isfinite(mse).all() or not torch.isfinite(kl).all():raise FloatingPointError('検証値が有限ではありません。')
            sums+=np.array([mse.sum().item(),kl.sum().item()]);count+=len(x)
    return {'mse':float(sums[0]/count),'kl':float(sums[1]/count)}

def prepare_training_tensors():
    variants=[]
    for parent,image in enumerate(train_images):
        for angle in ANGLES:
            tensor=to_tensor(final_transform(rotate_with_reflect(image,angle,verify_support=(parent==0))))
            if angle==0:assert torch.equal(tensor,all_train[parent])
            variants.append(tensor)
        if (parent+1)%50==0:print(f'回転学習入力の準備：{parent+1}/213',flush=True)
    rotated=torch.stack(variants)
    assert tuple(rotated.shape)==(2130,1,128,128)
    return {'selection_train':selection_train,'original':all_train,'rotated':rotated}

def train_task(spec,run_dir,tensors,initial_state):
    directory=run_dir/spec['task'];directory.mkdir(exist_ok=False)
    inputs=tensors[spec['dataset']]
    generator=torch.Generator().manual_seed(SEED+1000)
    loader=DataLoader(TensorDataset(inputs,torch.arange(len(inputs))),batch_size=BATCH_SIZE,
                      shuffle=True,generator=generator,drop_last=False,num_workers=0)
    assert len(inputs)%BATCH_SIZE!=1
    model=MODEL_FACTORIES[spec['model']](latent_dim=LATENT_DIM).to(DEVICE)
    model.load_state_dict(initial_state,strict=True);seed_everything(SEED)
    initial_hash=state_digest(copy_cpu_state(model))
    write_json(directory/'settings.json',{**spec,'contract_sha256':CONTRACT_SHA,'initial_state_sha256':initial_hash})
    optimizer=optim.Adam(model.parameters(),lr=LEARNING_RATE)
    rows=[];epochs=[];epoch=0;step=0;order_hash=hashlib.sha256();start_time=time.perf_counter()
    try:
        while step<spec['target_updates']:
            epoch+=1;epoch_sums=np.zeros(3);seen=0
            for batch_index,(x,indices) in enumerate(loader,1):
                model.train();x=x.to(DEVICE)
                beta=beta_at_update(step,spec['schedule_updates_per_epoch'])
                optimizer.zero_grad(set_to_none=True)
                out,mu,lv=model(x,sample=True)
                mse,kl,total=loss_terms(out,x,mu,lv,beta)
                if not all(torch.isfinite(v).all().item() for v in (mse,kl,total)):
                    raise FloatingPointError(f'{spec["task"]}：損失にNaN/inf')
                total.mean().backward()
                if any(p.grad is not None and not torch.isfinite(p.grad).all().item() for p in model.parameters()):
                    raise FloatingPointError('勾配にNaN/inf')
                optimizer.step();step+=1
                order_hash.update(indices.numpy().astype('<i8').tobytes())
                values=np.array([mse.sum().item(),kl.sum().item(),total.sum().item()])
                epoch_sums+=values;seen+=len(x)
                rows.append({'update':step,'epoch':epoch,'batch_in_epoch':batch_index,'batch_size':len(x),
                             'beta':beta,'train_mse':float(values[0]/len(x)),
                             'train_kl':float(values[1]/len(x)),'train_total':float(values[2]/len(x))})
                if step%67==0 or step==spec['target_updates']:
                    pd.DataFrame(rows).to_csv(directory/'step_history.csv',index=False)
                if step==spec['target_updates']:break
            record={'epoch':epoch,'update':step,'seen_images':seen,'complete_epoch':batch_index==len(loader),
                    'train_mse':float(epoch_sums[0]/seen),'train_kl':float(epoch_sums[1]/seen),
                    'train_total':float(epoch_sums[2]/seen),'beta_last':beta}
            if spec['group']=='selection':
                # 正常43枚のみ。テストデータは一切使わない。
                val=mean_reconstruction_metrics(model,selection_val)
                record.update(val_mse=val['mse'],val_kl=val['kl'])
            epochs.append(record)
            pd.DataFrame(epochs).to_csv(directory/'epoch_history.csv',index=False)
            if epoch==1 or epoch%10==0 or step==spec['target_updates']:
                print(f'{spec["label"]}：{step:,}/{spec["target_updates"]:,}更新、MSE={record["train_mse"]:.6f}、β={beta:.2f}',flush=True)
        frame=pd.DataFrame(rows)
        frame.to_csv(directory/'step_history.csv',index=False)
        weights=directory/'model_final.pth';torch.save(copy_cpu_state(model),weights)
        summary={**spec,'status':'completed','contract_sha256':CONTRACT_SHA,'updates':step,
                 'completed_full_epochs':epoch if batch_index==len(loader) else epoch-1,
                 'last_epoch_batches':batch_index,'seen_images':int(frame.batch_size.sum()),
                 'initial_state_sha256':initial_hash,'sample_order_sha256':order_hash.hexdigest(),
                 'model_sha256':sha256(weights),'seconds':time.perf_counter()-start_time}
        write_json(directory/'summary.json',summary)
        return summary
    except BaseException as exc:
        pd.DataFrame(rows).to_csv(directory/'step_history.csv',index=False)
        pd.DataFrame(epochs).to_csv(directory/'epoch_history.csv',index=False)
        write_json(directory/'failure.json',{'status':'failed_or_interrupted','completed_updates':step,
                                            'error':f'{type(exc).__name__}: {exc}'})
        raise
    finally:
        del model,optimizer
        if DEVICE.type=='cuda':torch.cuda.empty_cache()

## 4. 正式学習／保存済み実行の選択
**このセルを実行すると、初期値では7モデルの正式学習を開始します。長時間かかります。**

- 実行ごとの新しいフォルダを作り、既存モデルを上書きしません。
- 学習終了後の重みを保存します。検証やテストの良い時点を選ぶ方式ではありません。
- 中断した実行は未完了と記録します。途中のoptimizer状態からの再開機能は設けません。
- 再学習せず結果を読む場合：設定セルの `RUN_FULL_TRAINING=False` にし、`TRAINING_RUN_ID` を指定して上から実行してください。Noneなら最新の完了済み実行を選びます。
- データ識別情報・損失設定・学習計画が異なる実行は読み込みを拒否します。

In [ ]:
if RUN_FULL_TRAINING:
    RUN_DIR=OUTPUT_BASE/new_id();RUN_DIR.mkdir(parents=True,exist_ok=False)
    config={'contract':CONTRACT,'contract_sha256':CONTRACT_SHA,'status':'running','run_id':RUN_DIR.name,
            'created_at':datetime.now(ZoneInfo('Asia/Tokyo')).isoformat(),'device':str(DEVICE),
            'python':platform.python_version(),
            'versions':{v:importlib.metadata.version(v) for v in ['torch','torchvision','numpy','Pillow','scikit-image','scikit-learn']}}
    write_json(RUN_DIR/'config.json',config)
    write_json(RUN_DIR/'train_manifest.json',train_manifest);write_json(RUN_DIR/'split_manifest.json',split_manifest)
    model_table.to_csv(RUN_DIR/'model_table.csv',index=False)
    pd.DataFrame(PLAN).to_csv(RUN_DIR/'training_plan.csv',index=False)
    tensors=None
    try:
        tensors=prepare_training_tensors();initials={}
        write_json(RUN_DIR/'training_input_hashes.json',
                   {k:hashlib.sha256(v.numpy().tobytes()).hexdigest() for k,v in tensors.items()})
        for key in MODEL_KEYS:
            seed_everything(SEED);initial_model=MODEL_FACTORIES[key](latent_dim=LATENT_DIM)
            initials[key]=copy_cpu_state(initial_model);del initial_model
        summaries=[]
        for spec in PLAN:
            summaries.append(train_task(spec,RUN_DIR,tensors,initials[spec['model']]))
            pd.DataFrame(summaries).to_csv(RUN_DIR/'training_summary.csv',index=False)
        config['status']='completed';write_json(RUN_DIR/'config.json',config)
    except BaseException as exc:
        config.update(status='failed_or_interrupted',error=f'{type(exc).__name__}: {exc}')
        write_json(RUN_DIR/'config.json',config);raise
    finally:
        if tensors is not None:del tensors
else:
    if TRAINING_RUN_ID is not None:
        if Path(TRAINING_RUN_ID).name!=TRAINING_RUN_ID:raise ValueError('フォルダ名のみ指定してください。')
        RUN_DIR=OUTPUT_BASE/TRAINING_RUN_ID
    else:
        candidates=sorted((p for p in OUTPUT_BASE.glob('*') if (p/'config.json').is_file()
                           and read_json(p/'config.json').get('status')=='completed'),key=lambda p:p.name)
        if not candidates:raise FileNotFoundError('完了済みMSE実行がありません。学習を実行するか実行IDを確認してください。')
        RUN_DIR=candidates[-1]
    config=read_json(RUN_DIR/'config.json')
    if config.get('status')!='completed' or config.get('contract_sha256')!=CONTRACT_SHA or config.get('contract')!=CONTRACT:
        raise ValueError('未完了、または現在のデータ・MSE設定と異なる実行です。')

summaries={spec['task']:read_json(RUN_DIR/spec['task']/'summary.json') for spec in PLAN}
for spec in PLAN:
    summary=summaries[spec['task']]
    assert summary['status']=='completed' and summary['updates']==spec['target_updates']
    assert summary['contract_sha256']==CONTRACT_SHA
    assert sha256(RUN_DIR/spec['task']/'model_final.pth')==summary['model_sha256']
assert len({summaries[k]['initial_state_sha256'] for k in ROTATION_KEYS})==1
for spec in PLAN:
    h=pd.read_csv(RUN_DIR/spec['task']/'step_history.csv')
    assert np.array_equal(h['update'],np.arange(1,spec['target_updates']+1))
    expected=np.array([beta_at_update(i,spec['schedule_updates_per_epoch']) for i in range(len(h))])
    assert np.allclose(h.beta,expected,rtol=0,atol=1e-12)
rotated_history=pd.read_csv(RUN_DIR/'B_rotate100/step_history.csv')
matched_history=pd.read_csv(RUN_DIR/'A_normal_matched/step_history.csv')
assert np.array_equal(rotated_history.beta.to_numpy(),matched_history.beta.to_numpy())
print('学習実行ID：',RUN_DIR.name)
print('保存先：',RUN_DIR)
display(pd.DataFrame(summaries.values())[['task','updates','seen_images','status']])

## 5. 学習曲線（再学習せず保存ログから表示）
モデル選定の4モデルはエポックを横軸に、学習時のMSEと平均μからの検証MSEを区別します。
回転比較は更新回数を横軸にし、67更新ごとに画像数で重み付けした平均を表示します。
通常100エポックの曲線は700更新で終了します。各曲線の学習画像は同じ分布ではありません。
KLは潜在64次元の合計、画像間の平均。総損失にはその時点のβが掛かっています。

In [ ]:
# 表示先は毎回新規。既存の学習実行・評価結果は変更しません。
RESULT_DIR=RUN_DIR/'evaluations'/new_id();RESULT_DIR.mkdir(parents=True,exist_ok=False)
FIG_DIR=RESULT_DIR/'figures';FIG_DIR.mkdir()
WEIGHT_HASHES={k:sha256(RUN_DIR/k/'model_final.pth') for k in summaries}

def show_save(fig,name):
    fig.savefig(FIG_DIR/(name+'.png'),dpi=160,facecolor='white',bbox_inches='tight')
    plt.show();plt.close(fig)

fig,axes=plt.subplots(1,2,figsize=(14,5),constrained_layout=True)
for key in MODEL_KEYS:
    h=pd.read_csv(RUN_DIR/('selection_'+key)/'epoch_history.csv')
    axes[0].plot(h.epoch,h.train_mse,label=LABELS[key],color=COLORS[key])
    axes[1].plot(h.epoch,h.val_mse,label=LABELS[key],color=COLORS[key])
for ax,title in zip(axes,['学習MSE（サンプリング）','検証MSE（平均μ、正常43枚）']):
    ax.set(title=title,xlabel='エポック',ylabel='1画素あたり平均二乗誤差');ax.grid(ls='--',alpha=.4);ax.legend()
show_save(fig,'01_selection_mse')
fig,axes=plt.subplots(1,2,figsize=(14,5),constrained_layout=True)
for key in MODEL_KEYS:
    h=pd.read_csv(RUN_DIR/('selection_'+key)/'epoch_history.csv')
    axes[0].plot(h.epoch,h.train_total,label=LABELS[key],color=COLORS[key])
    axes[1].plot(h.epoch,h.train_kl,label=LABELS[key],color=COLORS[key])
for ax,title in zip(axes,['総損失：MSE＋βKL','KL項（潜在次元合計・画像平均）']):
    ax.set(title=title,xlabel='エポック',ylabel='学習画像あたり');ax.grid(ls='--',alpha=.4);ax.legend()
show_save(fig,'02_selection_total_kl')

def window_average(h,width=67):
    frame=h.copy();frame['window']=(frame['update']-1)//width
    out=[]
    for _,part in frame.groupby('window',sort=True):
        w=part.batch_size.to_numpy()
        row={'update':int(part['update'].iloc[-1]),'beta':float(part.beta.iloc[-1])}
        for metric in ['train_mse','train_kl','train_total']:
            row[metric]=float(np.average(part[metric],weights=w))
        out.append(row)
    return pd.DataFrame(out)

fig,axes=plt.subplots(1,3,figsize=(17,5),constrained_layout=True)
for key in ROTATION_KEYS:
    h=window_average(pd.read_csv(RUN_DIR/key/'step_history.csv'))
    for ax,metric,title in zip(axes,['train_total','train_mse','train_kl'],['総損失：MSE＋βKL','復元MSE','KL項']):
        ax.plot(h['update'],h[metric],label=LABELS[key],color=COLORS[key])
        ax.set(title=title,xlabel='更新回数',ylabel='学習画像あたり');ax.grid(ls='--',alpha=.4)
for ax in axes:ax.legend(fontsize=9)
show_save(fig,'03_rotation_training_curves')
fig,ax=plt.subplots(figsize=(10,4),constrained_layout=True)
for key in ROTATION_KEYS:
    h=pd.read_csv(RUN_DIR/key/'step_history.csv')
    ax.plot(h['update'],h.beta,label=LABELS[key],color=COLORS[key],alpha=.8)
ax.set(xlabel='更新回数',ylabel='β',title='通常6,700更新と回転拡張のβは一致');ax.legend();ax.grid(ls='--',alpha=.4)
show_save(fig,'04_beta_schedule')

## 6. 正常43枚で復元モデルを比較
全て100エポック終了時の重みを使い、平均μで復元します。
主な比較は検証MSE、SSIMと端子の見た目は補助です。testを使ってモデルを選びません。
この4モデルと、全213枚で学習する回転比較のHybridは別の実験です。

In [ ]:
def load_model(task):
    spec=next(s for s in PLAN if s['task']==task)
    path=RUN_DIR/task/'model_final.pth'
    if sha256(path)!=WEIGHT_HASHES[task]:raise ValueError('重みが変更されています。')
    model=MODEL_FACTORIES[spec['model']](latent_dim=LATENT_DIM).to(DEVICE)
    model.load_state_dict(torch.load(path,map_location=DEVICE,weights_only=True),strict=True)
    model.eval();return model

def reconstruct(model,inputs):
    reconstructed=np.empty((len(inputs),128,128),dtype=np.float32)
    kl_values=np.empty(len(inputs),dtype=np.float64)
    with torch.inference_mode():
        for start in range(0,len(inputs),EVAL_BATCH_SIZE):
            x=inputs[start:start+EVAL_BATCH_SIZE].to(DEVICE)
            out,mu,lv=model(x,sample=False)
            _,kl,_=loss_terms(out,x,mu,lv,0)
            if not torch.isfinite(out).all() or not torch.isfinite(kl).all():raise FloatingPointError('復元にNaN/inf')
            reconstructed[start:start+len(x)]=out[:,0].cpu().numpy()
            kl_values[start:start+len(x)]=kl.cpu().numpy()
    a=inputs[:,0].cpu().numpy()
    mse=np.square(a.astype(np.float64)-reconstructed.astype(np.float64)).mean(axis=(1,2))
    return {'reconstruction':reconstructed,'mse':mse,'kl':kl_values}

validation_cache={};validation_rows=[];validation_image_rows=[]
for key in MODEL_KEYS:
    model=load_model('selection_'+key)
    values=reconstruct(model,selection_val)
    values['ssim']=np.array([ssim(a,b,data_range=1.0,win_size=SSIM_WINDOW)
                            for a,b in zip(selection_val[:,0].numpy(),values['reconstruction'])])
    validation_cache[key]=values
    validation_rows.append({'model':key,'label':LABELS[key],'validation_mse':float(values['mse'].mean()),
                            'validation_ssim':float(values['ssim'].mean()),'validation_kl':float(values['kl'].mean())})
    for i,record in enumerate(split_manifest['validation']):
        validation_image_rows.append({'model':key,'path':record['path'],'mse':values['mse'][i],'ssim':values['ssim'][i]})
    del model
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
validation_comparison=pd.DataFrame(validation_rows)
validation_comparison.to_csv(RESULT_DIR/'validation_comparison.csv',index=False)
pd.DataFrame(validation_image_rows).to_csv(RESULT_DIR/'validation_image_metrics.csv',index=False)
display(validation_comparison)
print('検証平均MSEの最小候補：',validation_comparison.loc[validation_comparison.validation_mse.idxmin(),'label'])
print('復元品質の高いモデルが、異常検知でも最良とは限りません。')
fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax,col,title in zip(axes,['validation_mse','validation_ssim'],['検証MSE（小さい方が良い）','検証SSIM（大きい方が良い）']):
    bars=ax.bar(validation_comparison.label,validation_comparison[col],color=[COLORS[k] for k in MODEL_KEYS])
    ax.bar_label(bars,fmt='%.6f' if col=='validation_mse' else '%.4f',padding=3)
    ax.tick_params(axis='x',rotation=20);ax.set_title(title);ax.grid(axis='y',ls='--',alpha=.3)
    ax.set_ylim(min(0,float(validation_comparison[col].min())*1.2),max(1e-12,float(validation_comparison[col].max())*1.2))
show_save(fig,'05_validation_comparison')

In [ ]:
if MAX_VALIDATION_IMAGES is not None and (not isinstance(MAX_VALIDATION_IMAGES,int) or MAX_VALIDATION_IMAGES<=0):
    raise ValueError('MAX_VALIDATION_IMAGESは正の整数かNoneにしてください。')
count=len(selection_val) if MAX_VALIDATION_IMAGES is None else min(MAX_VALIDATION_IMAGES,len(selection_val))
for i in range(count):
    target=selection_val[i,0].numpy()
    fig,axes=plt.subplots(4,4,figsize=(15,13),constrained_layout=True)
    for row,key in enumerate(MODEL_KEYS):
        out=validation_cache[key]['reconstruction'][i]
        axes[row,0].imshow(target,cmap='gray',vmin=0,vmax=1)
        axes[row,1].imshow(out,cmap='gray',vmin=0,vmax=1)
        im=axes[row,2].imshow(np.abs(target-out),cmap='inferno',vmin=0,vmax=1)
        # 同じ位置の入力と復元を並べた拡大。新たな推論はしません。
        roi=np.concatenate([target[64:128,24:112],out[64:128,24:112]],axis=1)
        axes[row,3].imshow(roi,cmap='gray',vmin=0,vmax=1,interpolation='nearest')
        axes[row,0].set_ylabel(LABELS[key])
        axes[row,0].set_title('入力（復元目標）')
        axes[row,1].set_title(f'復元 MSE={validation_cache[key]["mse"][i]:.6f}\nSSIM={validation_cache[key]["ssim"][i]:.4f}')
        axes[row,2].set_title('絶対誤差 |入力−復元|')
        axes[row,3].set_title('端子の拡大：入力／復元')
        for ax in axes[row]:ax.set_xticks([]);ax.set_yticks([])
    fig.colorbar(im,ax=axes[:,2].tolist(),label='絶対誤差（全モデル共通）',shrink=.7)
    fig.suptitle(split_manifest['validation'][i]['path']+' ／正常の検証画像、平均μから復元')
    show_save(fig,f'validation_{i:03d}')
# 縮小の影響とモデルの復元誤差を区別する補助図。
path=ROOT/split_manifest['validation'][0]['path']
with Image.open(path) as im: original=im.convert('L').copy()
fig,axes=plt.subplots(1,2,figsize=(10,5),constrained_layout=True)
axes[0].imshow(original,cmap='gray',vmin=0,vmax=255);axes[0].set_title('元画像1024×1024')
axes[1].imshow(selection_val[0,0],cmap='gray',vmin=0,vmax=1,interpolation='nearest');axes[1].set_title('モデルに渡す128×128入力')
for ax in axes:ax.axis('off')
show_save(fig,'06_source_and_input')

## 7. 共通テスト100枚とランダム回転
正常60枚、異常4種類×10枚。元画像1枚につき、36〜324度の9角度から1角度をseed42で選びます。
0度は含めず、画像数は100枚のままです。同じ入力をHybridの3条件全てに使います。

前と同じ画像順・角度割り当てで比較しやすい設計ですが、新しいMSEモデルのAUROCは実行するまで分かりません。
合成回転を撮影方向の変化と仮定して元ラベルを維持します。特に配置ずれのラベル解釈、補完・欠け・既知角度という限界に注意してください。

In [ ]:
test_paths=sorted((ROOT/'data/transistor/test').glob('*/*.png'),key=lambda p:p.relative_to(ROOT/'data/transistor/test').as_posix())
test_images,test_tensor,test_manifest=load_images(test_paths)
records=[]
for path,record in zip(test_paths,test_manifest):
    category=path.parent.name
    records.append({**record,'relative_path':path.relative_to(ROOT/'data/transistor/test').as_posix(),
                    'category':category,'category_ja':CATEGORIES[category],'true_label':int(category!='good')})
image_table=pd.DataFrame(records)
assert image_table.category.value_counts().to_dict()=={'good':60,'bent_lead':10,'cut_lead':10,'damaged_case':10,'misplaced':10}
for key in ['file_sha256','input_sha256']:
    assert not {r[key] for r in train_manifest}&{r[key] for r in test_manifest}
assigned_angles=np.random.default_rng(ASSIGNMENT_SEED).choice(np.array(TEST_ANGLES,dtype=int),size=100,replace=True)
assert len(assigned_angles)==100 and np.all(assigned_angles!=0)
rotated_test=torch.stack([to_tensor(final_transform(rotate_with_reflect(im,int(angle),verify_support=True)))
                          for im,angle in zip(test_images,assigned_angles)])
assert test_tensor.shape==rotated_test.shape==(100,1,128,128)
scenario_inputs={'unrotated':test_tensor,'random_rotated':rotated_test}
assignment=image_table[['relative_path','category','category_ja','true_label']].copy()
assignment['assigned_angle']=assigned_angles
assignment.to_csv(RESULT_DIR/'angle_assignment.csv',index=False)
write_json(RESULT_DIR/'test_manifest.json',records)
eval_config={'status':'running','training_run':RUN_DIR.name,'training_contract_sha256':CONTRACT_SHA,
             'weight_sha256':WEIGHT_HASHES,'test_manifest_sha256':json_hash(records),
             'assignment_seed':ASSIGNMENT_SEED,'assignment_sha256':json_hash(assignment.to_dict('records')),
             'test_angles':list(TEST_ANGLES),'test_count':100,'top_k':TOP_K,'ssim_window':SSIM_WINDOW,
             'ssim_data_range':1.0,'anomaly_map':'1-SSIM, full map including boundaries, no clipping',
             'inference':'posterior mean','metric':'AUROC only; no threshold optimization',
             'scenario_tensor_sha256':{k:hashlib.sha256(v.numpy().tobytes()).hexdigest() for k,v in scenario_inputs.items()}}
write_json(RESULT_DIR/'evaluation_config.json',eval_config)
display(pd.crosstab(assignment.category_ja,assignment.assigned_angle).reindex(columns=TEST_ANGLES,fill_value=0))
print('テストはモデル選定に使用しません。学習済みHybrid 3条件の比較のみです。')

## 8. 平均推論・SSIM Top-130によるAUROC
異常スコアはSSIM誤差マップの大きい方から130**点**を平均します。独立した130パッチではありません。
復元MSEをそのまま異常スコアには使いません。AUROCは全100枚で計算し、閾値探索は行いません。
各画像の復元と誤差マップはこの評価で一度だけ計算し、以降の表示では同じ配列を使います。

In [ ]:
def anomaly_results(model,inputs):
    values=reconstruct(model,inputs)
    maps=np.empty_like(values['reconstruction'])
    structural=np.empty(len(inputs),dtype=np.float64)
    scores=np.empty(len(inputs),dtype=np.float64)
    for i,(a,b) in enumerate(zip(inputs[:,0].numpy(),values['reconstruction'])):
        score,ssim_map=ssim(a,b,data_range=1.0,win_size=SSIM_WINDOW,full=True)
        error=1.0-ssim_map
        if not np.isfinite(error).all():raise FloatingPointError('SSIM誤差にNaN/inf')
        structural[i]=score;maps[i]=error
        scores[i]=np.sort(error.ravel())[-TOP_K:].mean()
    return {**values,'ssim':structural,'error_map':maps,'scores':scores}

def verified_auroc(labels,scores):
    labels=np.asarray(labels);scores=np.asarray(scores,dtype=float)
    if not np.isfinite(scores).all():raise ValueError('スコアが有限ではありません。')
    value=float(roc_auc_score(labels,scores))
    normal=scores[labels==0];abnormal=scores[labels==1]
    rank=((abnormal[:,None]>normal).sum()+0.5*(abnormal[:,None]==normal).sum())/(len(normal)*len(abnormal))
    assert np.isclose(value,rank,rtol=0,atol=1e-12)
    return value

cache={};auroc_rows=[];per_image_rows=[]
y=image_table.true_label.to_numpy();normal_mask=(y==0)
for key in ROTATION_KEYS:
    model=load_model(key)
    for scenario,inputs in scenario_inputs.items():
        values=anomaly_results(model,inputs);cache[(key,scenario)]=values
        auroc_rows.append({'condition':key,'label':LABELS[key],'scenario':scenario,
                          'scenario_ja':SCENARIOS[scenario],'updates':summaries[key]['updates'],
                          'auroc':verified_auroc(y,values['scores']),
                          'normal_mse':float(values['mse'][normal_mask].mean()),
                          'normal_ssim':float(values['ssim'][normal_mask].mean()),
                          'normal_kl':float(values['kl'][normal_mask].mean())})
        for i,record in image_table.iterrows():
            per_image_rows.append({'condition':key,'scenario':scenario,'relative_path':record.relative_path,
                                   'category':record.category,'true_label':int(record.true_label),
                                   'angle':int(assigned_angles[i]) if scenario=='random_rotated' else 0,
                                   'score':float(values['scores'][i]),'mse':float(values['mse'][i]),
                                   'ssim':float(values['ssim'][i])})
    del model
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
comparison=pd.DataFrame(auroc_rows);score_table=pd.DataFrame(per_image_rows)
comparison.to_csv(RESULT_DIR/'auroc_comparison.csv',index=False)
score_table.to_csv(RESULT_DIR/'image_scores.csv',index=False)
display(comparison)
print('共通正常画像のMSE・SSIMは補助情報です。AUROCと同じ性能指標ではありません。')

In [ ]:
# まず、更新回数を揃えた2条件の4値を表示します。
matched_keys=('A_normal_matched','B_rotate100')
pivot=comparison.pivot(index='condition',columns='scenario',values='auroc')
fig,ax=plt.subplots(figsize=(10,6),constrained_layout=True)
x=np.arange(2);width=.36
for j,key in enumerate(matched_keys):
    bars=ax.bar(x+(j-.5)*width,[pivot.loc[key,s] for s in SCENARIOS],width,label=LABELS[key],color=COLORS[key])
    ax.bar_label(bars,fmt='%.5f',padding=4)
ax.set(xticks=x,xticklabels=list(SCENARIOS.values()),ylim=(0,1.08),ylabel='AUROC',title='6,700更新を揃えた比較')
ax.legend();ax.grid(axis='y',ls='--',alpha=.4)
show_save(fig,'07_matched_auroc')
fig,ax=plt.subplots(figsize=(11,6),constrained_layout=True)
x=np.arange(3);width=.36
for j,scenario in enumerate(SCENARIOS):
    bars=ax.bar(x+(j-.5)*width,[pivot.loc[k,scenario] for k in ROTATION_KEYS],width,label=SCENARIOS[scenario])
    ax.bar_label(bars,fmt='%.5f',padding=4)
ax.set(xticks=x,xticklabels=[LABELS[k] for k in ROTATION_KEYS],ylim=(0,1.08),ylabel='AUROC',title='通常100エポックも含む全3条件')
ax.legend();ax.grid(axis='y',ls='--',alpha=.4)
show_save(fig,'08_all_auroc')
for scenario in SCENARIOS:
    fig,ax=plt.subplots(figsize=(8,8),constrained_layout=True)
    for key in ROTATION_KEYS:
        scores=cache[(key,scenario)]['scores'];fpr,tpr,_=roc_curve(y,scores)
        ax.plot(fpr,tpr,color=COLORS[key],lw=2,label=f'{LABELS[key]} (AUROC={pivot.loc[key,scenario]:.5f})')
    ax.set(xlim=(0,1),ylim=(0,1.02),xlabel='偽陽性率 (False Positive Rate)',
           ylabel='真陽性率 (True Positive Rate)',title=SCENARIOS[scenario]+'：ROC曲線')
    ax.grid(ls='--',alpha=.4);ax.legend(loc='lower right')
    show_save(fig,'roc_'+scenario)
    joined=np.concatenate([cache[(key,scenario)]['scores'] for key in ROTATION_KEYS])
    lo,hi=float(joined.min()),float(joined.max())
    if lo==hi:lo-=1e-6;hi+=1e-6
    bins=np.linspace(lo,hi,40)
    for key in ROTATION_KEYS:
        scores=cache[(key,scenario)]['scores']
        fig,ax=plt.subplots(figsize=(10,6),constrained_layout=True)
        ax.hist(scores[y==0],bins=bins,color='green',alpha=.5,label='正常 (Normal)')
        ax.hist(scores[y==1],bins=bins,color='red',alpha=.5,label='異常 (Anomaly)')
        ax.set(xlabel='異常スコア（SSIM誤差の上位130点平均）',ylabel='画像数',
               title=LABELS[key]+' ／ '+SCENARIOS[scenario])
        ax.legend();ax.grid(ls='--',alpha=.4)
        show_save(fig,'hist_'+key+'_'+scenario)

## 9. 異常種類別の補足と共通正常画像の復元品質
種類別AUROCは正常60枚と、その異常種類10枚の比較です。全体AUROCの代わりにはせず、小標本の補足として扱います。
テスト正常画像の復元MSEは、3条件とも同じ60画像に対して計算します。

In [ ]:
category_rows=[]
for scenario in SCENARIOS:
    for category in CATEGORIES:
        if category=='good':continue
        mask=(image_table.category.to_numpy()==category)|normal_mask
        for key in ROTATION_KEYS:
            category_rows.append({'scenario':scenario,'category':category,'category_ja':CATEGORIES[category],
                                  'condition':key,'auroc':verified_auroc(y[mask],cache[(key,scenario)]['scores'][mask]),
                                  'normal_count':60,'anomaly_count':10})
category_comparison=pd.DataFrame(category_rows)
category_comparison.to_csv(RESULT_DIR/'category_auroc.csv',index=False)
for scenario in SCENARIOS:
    print(SCENARIOS[scenario])
    display(category_comparison[category_comparison.scenario==scenario].pivot(index='category_ja',columns='condition',values='auroc'))
fig,axes=plt.subplots(1,2,figsize=(13,5),constrained_layout=True)
for ax,scenario in zip(axes,SCENARIOS):
    part=comparison[comparison.scenario==scenario].set_index('condition').loc[list(ROTATION_KEYS)]
    bars=ax.bar([LABELS[k] for k in ROTATION_KEYS],part.normal_mse,color=[COLORS[k] for k in ROTATION_KEYS])
    ax.bar_label(bars,fmt='%.6f',padding=3)
    ax.set(title=SCENARIOS[scenario],ylabel='共通正常60枚の復元MSE')
    ax.set_ylim(0,float(part.normal_mse.max())*1.25+1e-9)
    ax.tick_params(axis='x',rotation=15);ax.grid(axis='y',ls='--',alpha=.4)
show_save(fig,'09_common_normal_mse')

## 10. 実際の入力・復元・SSIM誤差マップ
1画像につき行＝Hybridの3学習条件、列＝入力／復元／1−SSIMを表示します。
回転前後の両条件を表示し、ファイル名・割当角度・異常の種類・スコアを明記します。
誤差マップの色は全モデル・画像で共通の0〜2。明るいほど違いが大きく、欠陥の正解位置そのものではありません。
`MAX_IMAGES_PER_CATEGORY=None` なら全100画像×回転前後を表示します。表示条件はAUROCの集計に影響しません。

In [ ]:
if VISUAL_PATHS is not None:
    if len(set(VISUAL_PATHS))!=len(VISUAL_PATHS) or not set(VISUAL_PATHS).issubset(set(image_table.relative_path)):
        raise ValueError('表示画像名の重複・不存在を確認してください。')
    visual_indices=image_table.index[image_table.relative_path.isin(VISUAL_PATHS)].tolist()
else:
    if MAX_IMAGES_PER_CATEGORY is not None and (not isinstance(MAX_IMAGES_PER_CATEGORY,int) or MAX_IMAGES_PER_CATEGORY<=0):
        raise ValueError('表示枚数は正の整数かNoneにしてください。')
    visual_indices=[]
    for category in CATEGORIES:
        ids=image_table.index[image_table.category==category].tolist()
        visual_indices.extend(ids if MAX_IMAGES_PER_CATEGORY is None else ids[:MAX_IMAGES_PER_CATEGORY])
for i in visual_indices:
    record=image_table.iloc[i]
    for scenario,inputs in scenario_inputs.items():
        angle=int(assigned_angles[i]) if scenario=='random_rotated' else 0
        fig,axes=plt.subplots(3,3,figsize=(12,11),constrained_layout=True)
        for row,key in enumerate(ROTATION_KEYS):
            values=cache[(key,scenario)]
            axes[row,0].imshow(inputs[i,0],cmap='gray',vmin=0,vmax=1)
            axes[row,1].imshow(values['reconstruction'][i],cmap='gray',vmin=0,vmax=1)
            im=axes[row,2].imshow(values['error_map'][i],cmap='inferno',vmin=0,vmax=2)
            axes[row,0].set_title('入力画像');axes[row,0].set_ylabel(LABELS[key])
            axes[row,1].set_title(f'復元 MSE={values["mse"][i]:.6f}\n異常スコア={values["scores"][i]:.6f}')
            axes[row,2].set_title('SSIM誤差マップ (1−SSIM)')
            for ax in axes[row]:ax.set_xticks([]);ax.set_yticks([])
        fig.colorbar(im,ax=axes[:,2].tolist(),label='SSIM誤差（共通0〜2）',shrink=.7)
        fig.suptitle(f'{record.relative_path} ／正解：{record.category_ja} ／{angle}度\nMSE版の新規学習モデル、平均μから復元')
        show_save(fig,f'example_{i:03d}_{scenario}_{angle}deg')

## 11. 保存・出典・実行後の確認
学習ログ・重みは実行IDのフォルダ、今回の評価表・入力識別情報・図・配列はその下の `evaluations/評価ID/` へ保存します。
再評価は新しい評価IDで行います。既存の評価結果を上書きしません。

- 4モデルの構造・170/43分割：[従来の復元モデル比較](../復元モデル比較/261106transistor正常画像の復元に適したVAEモデルの比較.ipynb)
- 通常100周／回転100周・前処理：[従来の比較学習](../回転拡張の有用性検証/261105transistorの通常学習とreflect回転拡張の比較学習とAUROC評価.ipynb)
- 6,700更新・βの対応：[従来の更新回数合わせ](../回転拡張の有用性検証/261106transistorの更新回数を揃えた通常学習と回転拡張のAUROC比較.ipynb)
- 0度なしランダム回転：[従来の回転テスト](../回転拡張の有用性検証/261106transistorの検証画像の回転に対する異常検知性能の比較.ipynb)

**実行後に検討すること**

1. MSE＋βKL（β上限1）でも正常画像を復元できるか。KLが小さいことだけを良い結果としない。
2. 43枚の検証での平均MSE・SSIMと、端子や背景の見た目は一致するか。
3. 回転拡張の効果は、同じ100エポックの比較と、同じ6,700更新・βの比較でどう違うか。
4. 回転なしとランダム回転ありでAUROCの傾向が変わるか。従来のSSE版の結論を引き継がない。
5. 単一シード・単一の合成角度割り当て、学習で使った角度、補完・切り取り、既に見たテストでの探索的比較である点に注意する。
6. 復元品質と異常検知性能は別に評価する。Sparse VAEや実撮影条件への一般化は未検証。

β変更などを試す場合は設定が異なる新規実験として保存してください。今回のコードは設定を暗黙に調整しません。

In [ ]:
# 配列は評価時のものをそのまま保存。ここで再推論はしません。
np.savez_compressed(RESULT_DIR/'validation_cache.npz',inputs=selection_val.numpy(),
                    **{f'{key}_{name}':value for key,values in validation_cache.items() for name,value in values.items()})
np.savez_compressed(RESULT_DIR/'test_cache.npz',
                    **{f'inputs_{scenario}':value.numpy() for scenario,value in scenario_inputs.items()},
                    assigned_angles=assigned_angles,
                    **{f'{key}_{scenario}_{name}':value for (key,scenario),values in cache.items() for name,value in values.items()})
for task,digest in WEIGHT_HASHES.items():assert sha256(RUN_DIR/task/'model_final.pth')==digest
for record in train_manifest+test_manifest:assert sha256(ROOT/record['path'])==record['file_sha256']
eval_config.update(status='completed',validation_count=43,
                   result_files=['validation_comparison.csv','validation_image_metrics.csv','auroc_comparison.csv',
                                 'image_scores.csv','category_auroc.csv','angle_assignment.csv','validation_cache.npz','test_cache.npz'])
write_json(RESULT_DIR/'evaluation_config.json',eval_config)
print('学習モデル・ログ：',RUN_DIR)
print('今回の比較表・図・配列：',RESULT_DIR)
print('MSE版の実行結果です。従来SSEモデルの結果とは区別してください。')